In [ ]:
import copy
import time
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Subset
# import torch.nn.functional as F
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split

PREPROC_PATH = "./Dataset_preprocessed/scale_224"
BATCH_SIZE = 32
NUM_WORKERS = 0
RANDOM_STATE = 42
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD  = [0.229, 0.224, 0.225]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(p=0.4),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

full_train_dataset = datasets.ImageFolder(root=PREPROC_PATH, transform=train_transform)
full_val_dataset   = datasets.ImageFolder(root=PREPROC_PATH, transform=val_transform)

labels_all = np.array(full_train_dataset.targets)
indices    = np.arange(len(full_train_dataset))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=labels_all,
)

train_subset = Subset(full_train_dataset, train_idx)
val_subset   = Subset(full_val_dataset,   val_idx)

train_targets = labels_all[train_idx]
num_classes   = len(full_train_dataset.classes)
class_counts  = np.bincount(train_targets, minlength=num_classes)
class_weights = len(train_targets) / (num_classes * np.clip(class_counts, 1, None))
class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=device)

pin = torch.cuda.is_available()

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=pin)
val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pin)

print("Classes:", full_train_dataset.classes)
print("Train class counts:", class_counts)
print("Class weights:", class_weights)


Using device: cuda
Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Train class counts: [4500 3182 3078 3490]
Class weights: [0.79166667 1.11957888 1.15740741 1.02077364]


In [8]:
class Bench(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.block_1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.block_2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.block_3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Linear(256, 2048),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(2048, 2048),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(2048, num_classes),
        )

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, mean=0.0, std=0.05)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=0.05)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.block_1(x)
        x = self.block_2(x)
        x = self.block_3(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits = self.classifier(x)
        probas = torch.softmax(logits, dim=1)
        return logits, probas


In [9]:
torch.manual_seed(RANDOM_STATE)
model = Bench(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_t)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

In [10]:

NUM_EPOCHS = 20
PATIENCE   = 5
log = []
best_val_acc    = 0.0
best_model_wts  = None
epochs_no_improve = 0
start_time = time.time()

In [ ]:

for epoch in range(NUM_EPOCHS):
    model.train()
    run_loss, run_correct, run_total = 0.0, 0, 0

    for features, targets in train_loader:
        features, targets = features.to(device), targets.to(device)
        optimizer.zero_grad()
        logits, _ = model(features)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        run_loss += loss.item() * targets.size(0)
        run_correct += torch.eq(torch.argmax(logits, dim=1), targets).sum().item()
        run_total += targets.size(0)

    train_acc = 100.0 * run_correct / run_total
    train_loss = run_loss / run_total

    model.eval()
    val_loss_total, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for val_features, val_targets in val_loader:
            val_features, val_targets = val_features.to(device), val_targets.to(device)
            val_logits, _ = model(val_features)
            val_batch_loss = criterion(val_logits, val_targets)

            val_loss_total += val_batch_loss.item() * val_targets.size(0)
            val_correct += torch.eq(torch.argmax(val_logits, dim=1), val_targets).sum().item()
            val_total += val_targets.size(0)

    val_acc = 100.0 * val_correct / val_total
    val_loss = val_loss_total / val_total
    elapsed = (time.time() - start_time) / 60

    scheduler.step(val_loss)

    print('Epoch %02d/%02d | Train Acc: %.2f%% | Val Acc: %.2f%% | Train Loss: %.4f | Val Loss: %.4f | %.1f min'
          % (epoch+1, NUM_EPOCHS, train_acc, val_acc, train_loss, val_loss, elapsed), flush=True)

    log.append({'epoch': epoch+1, 'train_acc': train_acc, 'val_acc': val_acc,
                'train_loss': train_loss, 'val_loss': val_loss})

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
        print('  -> Best val acc: %.2f%%' % best_val_acc, flush=True)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print('Early stopping at epoch %d' % (epoch+1), flush=True)
            break

model.load_state_dict(best_model_wts)
torch.save(best_model_wts, 'bench_best.pth')
print('\nBest Val Acc: %.2f%%' % best_val_acc)
print('Total Time: %.2f min' % ((time.time() - start_time) / 60))


Epoch 01/20 | Train Acc: 52.70% | Val Acc: 47.52% | Train Loss: 1.0778 | Val Loss: 1.2476 | 10.2 min
  -> Best val acc: 47.52%
Epoch 02/20 | Train Acc: 68.55% | Val Acc: 69.66% | Train Loss: 0.7032 | Val Loss: 0.6607 | 20.1 min
  -> Best val acc: 69.66%
Epoch 03/20 | Train Acc: 71.35% | Val Acc: 73.37% | Train Loss: 0.6406 | Val Loss: 0.5953 | 30.0 min
  -> Best val acc: 73.37%
Epoch 04/20 | Train Acc: 72.74% | Val Acc: 73.90% | Train Loss: 0.5910 | Val Loss: 0.5973 | 40.5 min
  -> Best val acc: 73.90%
Epoch 05/20 | Train Acc: 74.19% | Val Acc: 72.78% | Train Loss: 0.5639 | Val Loss: 0.5728 | 51.1 min
Epoch 06/20 | Train Acc: 75.37% | Val Acc: 74.57% | Train Loss: 0.5404 | Val Loss: 0.5364 | 61.9 min
  -> Best val acc: 74.57%
Epoch 07/20 | Train Acc: 76.84% | Val Acc: 77.43% | Train Loss: 0.5098 | Val Loss: 0.5268 | 80.8 min
  -> Best val acc: 77.43%
Epoch 08/20 | Train Acc: 78.45% | Val Acc: 78.87% | Train Loss: 0.4896 | Val Loss: 0.4938 | 94.0 min
  -> Best val acc: 78.87%


In [ ]:
print("%-6s | %-10s | %-9s | %-11s | %-9s" % ("Epoch","Train Acc","Val Acc","Train Loss","Val Loss"))
print("-" * 55)
for r in log:
    print("%-6d | %-9.2f%% | %-8.2f%% | %-11.4f | %-9.4f"
          % (r['epoch'], r['train_acc'], r['val_acc'], r['train_loss'], r['val_loss']))
best = max(log, key=lambda x: x['val_acc'])
print("\nBest -> Epoch %d | Val Acc: %.2f%% | Val Loss: %.4f"
      % (best['epoch'], best['val_acc'], best['val_loss']))
